# 12 — ISIC 2019 Preprocessing

ISIC 2019'dan sadece melanoma (MEL) görüntülerini indirir,
HAM10000 ile aynı önişleme pipeline'ını uygular ve Drive'a kaydeder.

**Bunu bir kere çalıştır. Sonra 06-09 notebook'larını A100'de eğit.**

- Çıktı: `MyDrive/melanoma/X_isic2019_mel.npy` (~2 GB)
- Süre: ~20-30 dk (4 paralel thread)
- Runtime: **T4 GPU** (CPU-bound iş, A100 israfı olur)

In [1]:
# --- Setup ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)
else:
    subprocess.run(["git", "-C", str(project_root), "pull"], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import config
config.ensure_drive_dirs()
print("OK — Data dir:", config.DATA_DIR)

Mounted at /content/drive
OK — Data dir: /content/drive/MyDrive/melanoma/data


In [2]:
# --- X_all.npy'nin gerçek boyutunu oku (224 mı 448 mi?) ---
import numpy as np
from pathlib import Path

x_path = config.DATA_DIR / "X_all.npy"
X_tmp = np.load(x_path, mmap_mode='r')
ACTUAL_SIZE = X_tmp.shape[1]   # 224 veya 448
print(f"X_all.npy shape: {X_tmp.shape}")
print(f"ISIC 2019 da bu boyuta ({ACTUAL_SIZE}px) önişlenecek")

X_all.npy shape: (10015, 224, 224, 3)
ISIC 2019 da bu boyuta (224px) önişlenecek


In [3]:
# --- Kaggle auth ---
from pathlib import Path

kaggle_json = Path("/root/.kaggle/kaggle.json")
if not kaggle_json.exists():
    from google.colab import files
    print("kaggle.json yükle (kaggle.com → sağ üst profil → Settings → API → Create New Token)")
    uploaded = files.upload()
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.move(list(uploaded.keys())[0], str(kaggle_json))
    kaggle_json.chmod(0o600)
    print("Tamam.")
else:
    print("kaggle.json zaten var.")

kaggle.json yükle (kaggle.com → sağ üst profil → Settings → API → Create New Token)


Saving kaggle (1).json to kaggle (1).json
Tamam.


In [4]:
# --- ISIC 2019 indir (~10 GB, ~15 dk) ---
import subprocess
from pathlib import Path

ISIC_DIR = Path("/content/isic2019")
ISIC_DIR.mkdir(exist_ok=True)

csv_path = ISIC_DIR / "ISIC_2019_Training_GroundTruth.csv"
if not csv_path.exists():
    print("İndiriliyor (~10 GB, ~15 dk bekliyorsun)...")
    result = subprocess.run(
        ["kaggle", "datasets", "download",
         "-d", "andrewmvd/isic-2019",
         "-p", str(ISIC_DIR), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("İndirme başarısız — kaggle.json doğru mu?")
    print("İndirme tamamlandı.")
else:
    print("Zaten indirilmiş.")

print("Dosyalar:", [f.name for f in sorted(ISIC_DIR.iterdir())[:10]])

İndiriliyor (~10 GB, ~15 dk bekliyorsun)...
İndirme tamamlandı.
Dosyalar: ['ISIC_2019_Training_GroundTruth.csv', 'ISIC_2019_Training_Input', 'ISIC_2019_Training_Metadata.csv']


In [5]:
# --- Melanoma listesini çıkar + resim klasörünü bul ---
import pandas as pd, os
from pathlib import Path

gt = pd.read_csv(csv_path)
mel_ids = gt[gt["MEL"] == 1]["image"].tolist()
print(f"Toplam MEL görüntüsü: {len(mel_ids)}")

# Resim klasörünü bul
IMG_DIR = None
for candidate in [ISIC_DIR / "ISIC_2019_Training_Input", ISIC_DIR / "train", ISIC_DIR]:
    if candidate.exists() and any(candidate.glob("*.jpg")):
        IMG_DIR = candidate
        break
    for sub in candidate.glob("*/"):
        if any(sub.glob("*.jpg")):
            IMG_DIR = sub
            break
    if IMG_DIR:
        break

if IMG_DIR is None:
    for root, dirs, files in os.walk(str(ISIC_DIR)):
        if any(f.endswith(".jpg") for f in files):
            IMG_DIR = Path(root)
            break

mel_paths = [(img_id, IMG_DIR / f"{img_id}.jpg") for img_id in mel_ids
             if (IMG_DIR / f"{img_id}.jpg").exists()]
print(f"Bulunan: {len(mel_paths)}/{len(mel_ids)}")
print(f"Resim klasörü: {IMG_DIR}")

Toplam MEL görüntüsü: 4522
Bulunan: 4522/4522
Resim klasörü: /content/isic2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input


In [6]:
# --- Paralel önişleme (4 thread, ~20-30 dk) ---
import cv2, numpy as np, time, sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from src.preprocessing import preprocess_for_storage

N_WORKERS = 4
SIZE = ACTUAL_SIZE   # HAM10000 ile aynı boyut

def process_one(args):
    idx, img_id, img_path = args
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return idx, img_id, np.zeros((SIZE, SIZE, 3), dtype=np.uint8), False
    try:
        rgb, fb = preprocess_for_storage(img_bgr, size=SIZE)
        return idx, img_id, rgb, fb
    except Exception:
        return idx, img_id, np.zeros((SIZE, SIZE, 3), dtype=np.uint8), False

X_isic   = np.zeros((len(mel_paths), SIZE, SIZE, 3), dtype=np.uint8)
ids_isic = np.empty(len(mel_paths), dtype=object)
n_done = 0
n_fallback = 0
t0 = time.time()

tasks = [(i, img_id, img_path) for i, (img_id, img_path) in enumerate(mel_paths)]

with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(process_one, t): t[0] for t in tasks}
    for fut in as_completed(futures):
        idx, img_id, rgb, fb = fut.result()
        X_isic[idx]   = rgb
        ids_isic[idx] = img_id
        n_fallback += int(fb)
        n_done += 1
        if n_done % 200 == 0:
            elapsed = time.time() - t0
            rate = n_done / elapsed
            eta  = (len(mel_paths) - n_done) / rate
            print(f"  {n_done}/{len(mel_paths)}  |  {rate:.1f} img/s  |  ETA {eta/60:.1f} dk")

elapsed = time.time() - t0
print(f"\nBitti: {len(mel_paths)} görüntü, {elapsed/60:.1f} dakikada")
print(f"X_isic shape: {X_isic.shape}  dtype: {X_isic.dtype}")

  200/4522  |  15.2 img/s  |  ETA 4.7 dk
  400/4522  |  15.6 img/s  |  ETA 4.4 dk
  600/4522  |  15.8 img/s  |  ETA 4.1 dk
  800/4522  |  17.9 img/s  |  ETA 3.5 dk
  1000/4522  |  19.8 img/s  |  ETA 3.0 dk
  1200/4522  |  20.6 img/s  |  ETA 2.7 dk
  1400/4522  |  21.8 img/s  |  ETA 2.4 dk
  1600/4522  |  22.3 img/s  |  ETA 2.2 dk
  1800/4522  |  21.1 img/s  |  ETA 2.1 dk
  2000/4522  |  19.5 img/s  |  ETA 2.2 dk
  2200/4522  |  18.0 img/s  |  ETA 2.1 dk
  2400/4522  |  17.2 img/s  |  ETA 2.1 dk
  2600/4522  |  16.7 img/s  |  ETA 1.9 dk
  2800/4522  |  16.0 img/s  |  ETA 1.8 dk
  3000/4522  |  15.6 img/s  |  ETA 1.6 dk
  3200/4522  |  15.3 img/s  |  ETA 1.4 dk
  3400/4522  |  15.1 img/s  |  ETA 1.2 dk
  3600/4522  |  14.8 img/s  |  ETA 1.0 dk
  3800/4522  |  14.7 img/s  |  ETA 0.8 dk
  4000/4522  |  14.5 img/s  |  ETA 0.6 dk
  4200/4522  |  14.3 img/s  |  ETA 0.4 dk
  4400/4522  |  14.1 img/s  |  ETA 0.1 dk

Bitti: 4522 görüntü, 5.3 dakikada
X_isic shape: (4522, 224, 224, 3)  dtype: uin

In [7]:
# --- Drive'a kaydet ---
import numpy as np

OUT_X   = config.DATA_DIR / "X_isic2019_mel.npy"
OUT_IDS = config.DATA_DIR / "ids_isic2019_mel.npy"

print(f"Kaydediliyor: {OUT_X}  ({X_isic.nbytes/1e6:.0f} MB)")
np.save(OUT_X,   X_isic)
np.save(OUT_IDS, ids_isic)
print("Drive'a kaydedildi.")
print(f"X   : {OUT_X.stat().st_size/1e6:.0f} MB")
print(f"ids : {OUT_IDS.stat().st_size/1e3:.0f} KB")

Kaydediliyor: /content/drive/MyDrive/melanoma/data/X_isic2019_mel.npy  (681 MB)
Drive'a kaydedildi.
X   : 681 MB
ids : 73 KB


In [8]:
# --- Doğrulama: merge çalışıyor mu? ---
import importlib, src.data as _sd
importlib.reload(_sd)   # module cache sorununu önler
from src.data import load_arrays_extended

X_m, y_m, ids_m, idx_tr, idx_val, idx_test = load_arrays_extended(config.DATA_DIR)

n_mel  = int((y_m[idx_tr] == 1).sum())
n_nmel = int((y_m[idx_tr] == 0).sum())
print(f"Training melanoma   : {n_mel}")
print(f"Training non-mel    : {n_nmel}")
print(f"Oran                : 1 : {n_nmel/n_mel:.2f}")
print(f"Val/test            : {len(idx_val)} / {len(idx_test)}  (saf HAM10000)")
print(f"Toplam X shape      : {X_m.shape}")
print("\nHAZIR. Şimdi 06_resnet50.ipynb veya 07_efficientnet_b3.ipynb'i A100'de çalıştır.")

  copying X_all.npy (1507.5 MB) Drive -> local SSD
  copying y_all.npy (0.1 MB) Drive -> local SSD
  copying ids_all.npy (0.2 MB) Drive -> local SSD
  copying idx_train.npy (0.1 MB) Drive -> local SSD
  copying idx_val.npy (0.0 MB) Drive -> local SSD
  copying idx_test.npy (0.0 MB) Drive -> local SSD
  copying X_isic2019_mel.npy (680.7 MB) Drive -> local SSD
  copying ids_isic2019_mel.npy (0.1 MB) Drive -> local SSD
[load_arrays_extended] HAM10000 train mel: 779  +  ISIC2019 mel: 4522  =  5301 total mel in training
  Extended X shape: (14537, 224, 224, 3)
Training melanoma   : 5301
Training non-mel    : 6230
Oran                : 1 : 1.18
Val/test            : 1503 / 1503  (saf HAM10000)
Toplam X shape      : (14537, 224, 224, 3)

HAZIR. Şimdi 06_resnet50.ipynb veya 07_efficientnet_b3.ipynb'i A100'de çalıştır.
